方針は、ver07 をベースにしつつ、多段化だけでは改善しなかった原因を「候補空間の広さ」と「action候補の汚染」にあるとみなし、そこを直しています。

ver08 の中核改善は4点です。

- Stage0: family 推定 を追加. いきなり全 action から選ばせず、まず粗い family に落とす
- retrieval 後に family で候補を再圧縮. camera 系 instruction に style / quantity 系 action を見せない
- action は family 制約つきで選択
- params は LLM に自由生成させず、候補例からの選択＋丸めを強化

つまり、ver07 の
```json
instruction
  ↓
retrieval
  ↓
all local actions
  ↓
LLM multi-stage
```
を
```json
instruction
  ↓
retrieval
  ↓
all local actions
  ↓
LLM multi-stage
```
に変える。

In [1]:
#!/usr/bin/env python
# coding: utf-8

# ============================================================
# ver08
# ============================================================
# 背景
# ------------------------------------------------------------
# このコードの目的は、
#   instruction -> task planning JSON
# を、GT（annotations_gt_task_ver09.json）に強く制約された形で
# 安定出力すること。
#
# ver07 の問題：
# - multi-stage 化（action → target → params）は導入した
# - しかし、最初に見せる候補空間がまだ広すぎた
# - そのため camera 系 instruction でも style 系や quantity 系 action が混入し、
#   "apply_style", "add_object" などの代表解に潰れた
#
# ver08 の改善方針：
# 1. まず instruction から coarse family を選ぶ
# 2. retrieval した候補も family で再圧縮する
# 3. action は family に属するものだけから選ぶ
# 4. target / constraints / params も action に強く従属させる
#
# これにより、
#   「候補が広すぎて間違う」
# という ver07 の根本原因を潰す。


# ============================================================
# 0. import
# ============================================================
import json
import re
from difflib import SequenceMatcher
from typing import Any, Dict, List

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm


# ============================================================
# 1. 設定
# ============================================================
# 背景意図:
# - ver07 のパスや保存先を踏襲し、比較しやすくする
DATA_ROOT_DIR = "/workspace/data"
GT_PATH = f"{DATA_ROOT_DIR}/annotations_gt_task_ver09.json"

# 背景意図:
# - JSON 出力安定性を重視
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

LOCAL_FILES_ONLY = False

# 背景意図:
# - retrieval は多すぎるとノイズ、少なすぎると欠落
TOP_K_RETRIEVE = 16
TOP_K_EXAMPLES = 4

# 背景意図:
# - stage ごとに短く出させる
MAX_NEW_TOKENS_STAGE0 = 80
MAX_NEW_TOKENS_STAGE1 = 80
MAX_NEW_TOKENS_STAGE2 = 140
MAX_NEW_TOKENS_STAGE3 = 120

OUTPUT_PATH = "/workspace/notebook/prediction_results_ver08.json"


# ============================================================
# 2. 文字列正規化・類似度
# ============================================================
def normalize_text(s: str) -> str:
    """
    背景意図:
    - retrieval / 丸め込み / 評価のすべてで同じ正規化を使う
    - underscore, 記号, 大文字小文字差を吸収する
    """
    if s is None:
        return ""
    s = str(s).lower().strip()
    s = s.replace("_", " ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def text_similarity(a: str, b: str) -> float:
    """
    背景意図:
    - 軽量な retrieval と候補丸め込みの両方で使う
    - character 類似 + token overlap を混ぜる
    """
    a_n = normalize_text(a)
    b_n = normalize_text(b)

    if not a_n and not b_n:
        return 1.0
    if not a_n or not b_n:
        return 0.0

    char_sim = SequenceMatcher(None, a_n, b_n).ratio()

    a_tokens = set(a_n.split())
    b_tokens = set(b_n.split())
    jaccard = len(a_tokens & b_tokens) / max(1, len(a_tokens | b_tokens))

    return 0.45 * char_sim + 0.55 * jaccard


def nearest_choice(value: str, candidates: List[str]) -> str:
    """
    背景意図:
    - LLM が候補外の近い文字列を出したときに、最も近い候補へ戻す
    """
    if not candidates:
        return ""

    value_n = normalize_text(value)

    # 完全一致優先
    for c in candidates:
        if normalize_text(c) == value_n:
            return c

    best = None
    best_score = -1.0
    for c in candidates:
        s = text_similarity(value, c)
        if s > best_score:
            best_score = s
            best = c

    return best if best is not None else ""


# ============================================================
# 3. GT primary task 抽出
# ============================================================
PRIMARY_ACTION_PRIORITY = [
    "replace_background",
    "replace_object",
    "add_object",
    "increase_amount",
    "change_color",
    "remove_object",
    "edit_motion",
    "edit_expression",
    "change_camera_angle",
    "zoom_in",
    "zoom_out",
    "dolly_in",
    "orbit_camera",
    "apply_style",
    "add_effect",
    # 補助
    "preserve_foreground",
    "preserve_objects",
    "preserve_identity",
    "preserve_focus",
    "preserve_framing",
    "preserve_layout",
    "preserve_material_appearance",
    "align_replacement",
    "match_appearance",
    "match_lighting",
    "match_background_camera_properties",
    "match_effect_lighting",
    "match_scene_interaction",
    "stabilize_instances",
    "stabilize_edit",
    "stabilize_motion",
    "stabilize_style",
    "stabilize_effect",
    "stabilize_composite",
    "stabilize_inpaint",
    "refine_mask",
    "blend_instances",
    "inpaint_background",
    "adjust_perspective",
    "track_effect",
    "enhance_style_details",
]
PRIMARY_ACTION_RANK = {a: i for i, a in enumerate(PRIMARY_ACTION_PRIORITY)}


def extract_primary_task(tasks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    背景意図:
    - GT は tasks を複数持つ
    - 今回の予測は単一 task なので、GT 側も単一に縮約して比較する
    - tasks[0] 固定ではなく、主編集 action 優先
    """
    if not tasks:
        return {"action": "", "target": "", "constraints": [], "params": {}}

    ranked = []
    for idx, t in enumerate(tasks):
        action = normalize_text(t.get("action", ""))
        rank = PRIMARY_ACTION_RANK.get(action, 9999)
        ranked.append((rank, idx, t))

    ranked.sort(key=lambda x: (x[0], x[1]))
    primary = ranked[0][2]

    return {
        "action": primary.get("action", ""),
        "target": primary.get("target", ""),
        "constraints": primary.get("constraints", []),
        "params": primary.get("params", {}),
    }


# ============================================================
# 4. family 定義
# ============================================================
# 背景意図:
# - ver07 の最大問題は action 候補が広すぎたこと
# - 先に family に落として候補空間を切る
FAMILY_TO_ACTIONS = {
    "camera_motion_family": ["zoom_in", "zoom_out", "dolly_in", "orbit_camera"],
    "camera_angle_family": ["change_camera_angle"],
    "motion_edit_family": ["edit_motion", "edit_expression"],
    "quantity_family": ["add_object", "increase_amount"],
    "background_change_family": ["replace_background"],
    "style_family": ["apply_style"],
    "color_family": ["change_color"],
    "instance_replace_family": ["replace_object", "remove_object"],
    "effect_family": ["add_effect"],
    "other_family": []
}

ACTION_TO_FAMILY = {}
for fam, acts in FAMILY_TO_ACTIONS.items():
    for act in acts:
        ACTION_TO_FAMILY[act] = fam

ALL_FAMILIES = list(FAMILY_TO_ACTIONS.keys())


def family_from_action(action: str) -> str:
    return ACTION_TO_FAMILY.get(action, "other_family")


# ============================================================
# 5. GT候補辞書構築
# ============================================================
def freeze_value(v: Any) -> Any:
    """
    背景意図:
    - dict/list を set に入れられる形へ変換
    """
    if isinstance(v, dict):
        return tuple(sorted((k, freeze_value(val)) for k, val in v.items()))
    if isinstance(v, list):
        return tuple(freeze_value(x) for x in v)
    return v


def unfreeze_value(v: Any) -> Any:
    """
    背景意図:
    - freeze した値を JSON で扱える形に戻す
    """
    if isinstance(v, tuple):
        if len(v) > 0 and all(isinstance(x, tuple) and len(x) == 2 and isinstance(x[0], str) for x in v):
            return {k: unfreeze_value(val) for k, val in v}
        return [unfreeze_value(x) for x in v]
    return v


def target_to_hashable(tgt: Any):
    """
    背景意図:
    - target は str / list 混在があり得る
    - 内部では tuple に統一して set に入れる
    """
    if tgt is None:
        return tuple()
    if isinstance(tgt, list):
        return tuple(str(x) for x in tgt)
    if isinstance(tgt, str):
        return (tgt,)
    return (str(tgt),)


def build_candidate_dictionary(dataset: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    背景意図:
    - GT 全体から候補空間を構築する
    - action, family, target, constraints, params の関係を保持する
    """
    action_set = set()
    family_set = set()

    action_to_targets = {}
    action_to_constraints = {}
    action_to_params = {}
    family_to_actions = {}

    for item in dataset:
        primary = extract_primary_task(item.get("tasks", []))

        act = primary.get("action", "")
        fam = family_from_action(act)
        tgt = target_to_hashable(primary.get("target", ""))
        cons = tuple(primary.get("constraints", []))
        prm = freeze_value(primary.get("params", {}))

        if not act:
            continue

        action_set.add(act)
        family_set.add(fam)

        family_to_actions.setdefault(fam, set()).add(act)
        action_to_targets.setdefault(act, set()).add(tgt)
        action_to_constraints.setdefault(act, set()).add(cons)
        action_to_params.setdefault(act, set()).add(prm)

    return {
        "families": sorted(list(family_set)),
        "actions": sorted(list(action_set)),
        "family_to_actions": {
            k: sorted(list(v)) for k, v in family_to_actions.items()
        },
        "action_to_targets": {
            k: [list(x) for x in sorted(list(v), key=lambda z: str(z))]
            for k, v in action_to_targets.items()
        },
        "action_to_constraints": {
            k: [list(x) for x in sorted(list(v), key=lambda z: str(z))]
            for k, v in action_to_constraints.items()
        },
        "action_to_params": {
            k: [unfreeze_value(x) for x in sorted(list(v), key=lambda z: str(z))]
            for k, v in action_to_params.items()
        }
    }


# ============================================================
# 6. Retrieval
# ============================================================
def retrieve_topk_examples(
    instruction: str,
    dataset: List[Dict[str, Any]],
    top_k: int = TOP_K_RETRIEVE
) -> List[Dict[str, Any]]:
    """
    背景意図:
    - 全 GT を毎回見せると長すぎる
    - instruction に近い例だけを候補圧縮に使う
    """
    scored = []
    for item in dataset:
        s = text_similarity(instruction, item["instruction"])
        scored.append((s, item))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [x[1] for x in scored[:top_k]]


def build_local_candidates(retrieved: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    背景意図:
    - retrieval 上位から local 候補集合を作る
    - ver07 ではここに family 制約がなかった
    - ver08 では family 推定後にさらに再圧縮する
    """
    actions = set()
    families = set()
    targets = set()
    constraints_pool = set()
    params_pool = set()

    for item in retrieved:
        primary = extract_primary_task(item.get("tasks", []))
        act = primary.get("action", "")
        fam = family_from_action(act)
        tgt = primary.get("target", "")
        cons = primary.get("constraints", [])
        prm = primary.get("params", {})

        if act:
            actions.add(act)
            families.add(fam)

        if tgt:
            targets.add(tuple(target_to_hashable(tgt)))

        if isinstance(cons, list):
            constraints_pool.add(tuple(cons))
        else:
            constraints_pool.add((str(cons),))

        params_pool.add(freeze_value(prm))

    return {
        "families": sorted(list(families)),
        "actions": sorted(list(actions)),
        "targets": [list(x) for x in sorted(list(targets), key=lambda z: str(z))],
        "constraints_pool": [list(x) for x in sorted(list(constraints_pool), key=lambda z: str(z))],
        "params_pool": [unfreeze_value(x) for x in sorted(list(params_pool), key=lambda z: str(z))]
    }


def restrict_candidates_by_family(local_candidates: Dict[str, Any], family: str, global_dict: Dict[str, Any]) -> Dict[str, Any]:
    """
    背景意図:
    - ver08 の核心
    - family 決定後、action 候補をその family に属するものだけへ絞る
    - action 候補が狭まれば、target / params も自然に狭まる
    """
    allowed_actions = set(global_dict["family_to_actions"].get(family, []))
    if not allowed_actions:
        allowed_actions = set(local_candidates["actions"])

    actions = [a for a in local_candidates["actions"] if a in allowed_actions]
    if not actions:
        actions = list(sorted(allowed_actions)) if allowed_actions else local_candidates["actions"]

    targets = []
    constraints_pool = []
    params_pool = []

    seen_targets = set()
    seen_constraints = set()
    seen_params = set()

    for act in actions:
        for tgt in global_dict["action_to_targets"].get(act, []):
            key = tuple(tgt)
            if key not in seen_targets:
                seen_targets.add(key)
                targets.append(tgt)

        for cons in global_dict["action_to_constraints"].get(act, []):
            key = tuple(cons)
            if key not in seen_constraints:
                seen_constraints.add(key)
                constraints_pool.append(cons)

        for prm in global_dict["action_to_params"].get(act, []):
            key = str(freeze_value(prm))
            if key not in seen_params:
                seen_params.add(key)
                params_pool.append(prm)

    return {
        "families": [family],
        "actions": actions,
        "targets": targets,
        "constraints_pool": constraints_pool,
        "params_pool": params_pool
    }


# ============================================================
# 7. few-shot
# ============================================================
def build_fewshot_examples(retrieved: List[Dict[str, Any]], k: int = TOP_K_EXAMPLES) -> str:
    """
    背景意図:
    - retrieval 上位の具体例を few-shot として見せる
    - ただし自由生成を促すのではなく、出力形の見本として使う
    """
    chunks = []
    for item in retrieved[:k]:
        primary = extract_primary_task(item.get("tasks", []))
        example = {
            "tasks": [{
                "action": primary["action"],
                "target": primary["target"],
                "constraints": primary["constraints"],
                "params": primary["params"]
            }]
        }

        chunks.append(
            f"""Instruction:
{item["instruction"]}

Output:
{json.dumps(example, ensure_ascii=False, indent=2)}
"""
        )

    return "\n".join(chunks)


# ============================================================
# 8. モデルロード
# ============================================================
def load_model(model_name: str = MODEL_NAME, local_files_only: bool = LOCAL_FILES_ONLY):
    """
    背景意図:
    - ver07 と同じ運用で差し替えやすくする
    - deprecated 警告を避けるため dtype を使う
    """
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        local_files_only=local_files_only
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map="auto",
        local_files_only=local_files_only
    )
    return tokenizer, model


# ============================================================
# 9. JSON抽出
# ============================================================
def extract_json_object(text: str) -> Dict[str, Any]:
    """
    背景意図:
    - LLM 出力の前後ノイズを許容しつつ JSON 部分だけ拾う
    """
    start = text.find("{")
    end = text.rfind("}") + 1
    if start == -1 or end == 0:
        return {}

    json_str = text[start:end]
    try:
        return json.loads(json_str)
    except Exception:
        return {}


# ============================================================
# 10. flatten / params 類似
# ============================================================
def flatten_json(obj: Any, prefix: str = "") -> Dict[str, str]:
    out = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_prefix = f"{prefix}.{k}" if prefix else k
            out.update(flatten_json(v, new_prefix))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            new_prefix = f"{prefix}[{i}]"
            out.update(flatten_json(v, new_prefix))
    else:
        out[prefix] = str(obj)
    return out


def nearest_params(pred_params: Dict[str, Any], param_candidates: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    背景意図:
    - params は action ごとの候補例から最も近いものを選ぶ
    - 自由生成より安定させる
    """
    if not param_candidates:
        return {}

    pred_flat = flatten_json(pred_params)

    best = None
    best_score = -1.0

    for cand in param_candidates:
        cand_flat = flatten_json(cand)

        if not pred_flat and not cand_flat:
            return cand

        total = 0.0
        count = 0

        for ck, cv in cand_flat.items():
            local_best = 0.0
            for pk, pv in pred_flat.items():
                key_sim = text_similarity(ck, pk)
                val_sim = text_similarity(cv, pv)
                local_best = max(local_best, 0.5 * key_sim + 0.5 * val_sim)
            total += local_best
            count += 1

        score = total / max(1, count)
        if score > best_score:
            best_score = score
            best = cand

    return best if best is not None else {}


# ============================================================
# 11. 汎用LLM実行
# ============================================================
def run_llm_json(prompt: str, tokenizer, model, max_new_tokens: int) -> Dict[str, Any]:
    """
    背景意図:
    - 各 stage の共通LLM呼び出し
    - JSON only を前提に最小共通化する
    """
    if hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        text = prompt

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = extract_json_object(decoded)
    return pred if pred else {}


# ============================================================
# 12. Stage0: family 推定
# ============================================================
def build_prompt_stage0_family(instruction: str, local_candidates: Dict[str, Any]) -> str:
    """
    背景意図:
    - ver08 の最重要追加
    - 先に粗い family を当てて、action 候補空間を切る
    """
    return f"""
You are a strict classifier.

Select ONLY ONE family from the candidate list.

Instruction:
{instruction}

Candidate families:
{json.dumps(local_candidates["families"], ensure_ascii=False)}

Rules:
- MUST choose one family from the candidate list
- Do NOT invent a new family
- Output JSON only

Output:
{{"family": "..."}}
""".strip()


def run_stage0_family(instruction: str, tokenizer, model, local_candidates: Dict[str, Any]) -> str:
    pred = run_llm_json(
        build_prompt_stage0_family(instruction, local_candidates),
        tokenizer,
        model,
        MAX_NEW_TOKENS_STAGE0
    )
    raw_family = pred.get("family", "")
    family = nearest_choice(raw_family, local_candidates["families"])
    return family if family else "other_family"


# ============================================================
# 13. Stage1: action
# ============================================================
def build_prompt_stage1_action(instruction: str, local_candidates: Dict[str, Any]) -> str:
    return f"""
You are a strict selector.

Select ONLY ONE action from the candidate list.

Instruction:
{instruction}

Candidate actions:
{json.dumps(local_candidates["actions"], ensure_ascii=False)}

Rules:
- MUST choose one action from the candidate list
- Do NOT invent a new action
- Output JSON only

Output:
{{"action": "..."}}
""".strip()


def run_stage1_action(instruction: str, tokenizer, model, local_candidates: Dict[str, Any]) -> str:
    pred = run_llm_json(
        build_prompt_stage1_action(instruction, local_candidates),
        tokenizer,
        model,
        MAX_NEW_TOKENS_STAGE1
    )
    raw_action = pred.get("action", "")
    return nearest_choice(raw_action, local_candidates["actions"])


# ============================================================
# 14. Stage2: target / constraints
# ============================================================
def build_prompt_stage2_target(instruction: str, action: str, target_candidates: List[Any], constraint_candidates: List[Any]) -> str:
    return f"""
You are a strict selector.

Instruction:
{instruction}

Chosen action:
{action}

Candidate targets:
{json.dumps(target_candidates, ensure_ascii=False)}

Candidate constraints:
{json.dumps(constraint_candidates, ensure_ascii=False)}

Rules:
- MUST choose target from Candidate targets
- constraints should be chosen from Candidate constraints if relevant
- Output JSON only

Output:
{{
  "target": "...",
  "constraints": ["..."]
}}
""".strip()


def run_stage2_target_constraints(
    instruction: str,
    action: str,
    global_dict: Dict[str, Any],
    tokenizer,
    model
):
    target_candidates = global_dict["action_to_targets"].get(action, [])
    constraint_candidates = global_dict["action_to_constraints"].get(action, [])

    pred = run_llm_json(
        build_prompt_stage2_target(instruction, action, target_candidates, constraint_candidates),
        tokenizer,
        model,
        MAX_NEW_TOKENS_STAGE2
    )

    raw_target = pred.get("target", "")
    raw_constraints = pred.get("constraints", [])

    norm_target = nearest_choice(raw_target, target_candidates)

    best_constraints = []
    best_score = -1.0

    for cand in constraint_candidates:
        total = 0.0
        if not cand and not raw_constraints:
            best_constraints = []
            best_score = 1.0
            break

        for c in cand:
            local_best = 0.0
            for rc in raw_constraints:
                local_best = max(local_best, text_similarity(c, rc))
            total += local_best

        score = total / max(1, len(cand))
        if score > best_score:
            best_score = score
            best_constraints = cand

    return norm_target, best_constraints


# ============================================================
# 15. Stage3: params
# ============================================================
def build_prompt_stage3_params(instruction: str, action: str, params_candidates: List[Dict[str, Any]]) -> str:
    return f"""
You are a strict selector.

Instruction:
{instruction}

Chosen action:
{action}

Candidate params examples:
{json.dumps(params_candidates, ensure_ascii=False, indent=2)}

Rules:
- Choose the closest params example for this instruction
- Output JSON only

Output:
{{"params": {{}}}}
""".strip()


def run_stage3_params(
    instruction: str,
    action: str,
    global_dict: Dict[str, Any],
    tokenizer,
    model
) -> Dict[str, Any]:
    params_candidates = global_dict["action_to_params"].get(action, [])

    pred = run_llm_json(
        build_prompt_stage3_params(instruction, action, params_candidates),
        tokenizer,
        model,
        MAX_NEW_TOKENS_STAGE3
    )

    raw_params = pred.get("params", {})
    if not isinstance(raw_params, dict):
        raw_params = {}

    return nearest_params(raw_params, params_candidates)


# ============================================================
# 16. ver08 main prediction
# ============================================================
def generate_task_json_ver08(
    instruction: str,
    dataset: List[Dict[str, Any]],
    global_dict: Dict[str, Any],
    tokenizer,
    model
) -> Dict[str, Any]:
    """
    背景意図:
    - ver08 の本体
    - retrieval → family → restricted action space → target/constraints → params
    """

    # --------------------------------------------------------
    # Step A: retrieval
    # --------------------------------------------------------
    retrieved = retrieve_topk_examples(instruction, dataset, TOP_K_RETRIEVE)
    local_candidates = build_local_candidates(retrieved)

    # --------------------------------------------------------
    # Step B: Stage0 family
    # --------------------------------------------------------
    family = run_stage0_family(instruction, tokenizer, model, local_candidates)

    # --------------------------------------------------------
    # Step C: family で候補再圧縮
    # --------------------------------------------------------
    restricted_candidates = restrict_candidates_by_family(local_candidates, family, global_dict)

    # family 制約で候補が空になると困るため fallback
    if not restricted_candidates["actions"]:
        restricted_candidates = local_candidates

    # --------------------------------------------------------
    # Step D: Stage1 action
    # --------------------------------------------------------
    action = run_stage1_action(instruction, tokenizer, model, restricted_candidates)

    # action fallback
    if not action:
        action = nearest_choice("", restricted_candidates["actions"])

    # --------------------------------------------------------
    # Step E: Stage2 target / constraints
    # --------------------------------------------------------
    target, constraints = run_stage2_target_constraints(
        instruction, action, global_dict, tokenizer, model
    )

    # --------------------------------------------------------
    # Step F: Stage3 params
    # --------------------------------------------------------
    params = run_stage3_params(
        instruction, action, global_dict, tokenizer, model
    )

    return {
        "tasks": [{
            "action": action,
            "target": target,
            "constraints": constraints,
            "params": params
        }]
    }


# ============================================================
# 17. 評価（ver07流用）
# ============================================================
def score_constraints(pred_constraints: List[str], gt_constraints: List[str]) -> float:
    if not pred_constraints and not gt_constraints:
        return 1.0
    if not gt_constraints:
        return 1.0 if not pred_constraints else 0.5

    total = 0.0
    for g in gt_constraints:
        best = 0.0
        for p in pred_constraints:
            best = max(best, text_similarity(g, p))
        total += best

    return total / len(gt_constraints)


def score_params(pred_params: Dict[str, Any], gt_params: Dict[str, Any]) -> float:
    gt_flat = flatten_json(gt_params)
    pred_flat = flatten_json(pred_params)

    if not gt_flat and not pred_flat:
        return 1.0
    if not gt_flat:
        return 0.5

    total = 0.0
    for gk, gv in gt_flat.items():
        best = 0.0
        for pk, pv in pred_flat.items():
            key_sim = text_similarity(gk, pk)
            val_sim = text_similarity(gv, pv)
            best = max(best, 0.5 * key_sim + 0.5 * val_sim)
        total += best

    return total / len(gt_flat)


def score_prediction(pred: Dict[str, Any], gt_item: Dict[str, Any]) -> Dict[str, float]:
    gt_task = extract_primary_task(gt_item["tasks"])
    pred_task = pred["tasks"][0]

    scores = {
        "action": text_similarity(pred_task.get("action", ""), gt_task.get("action", "")),
        "target": text_similarity(pred_task.get("target", ""), gt_task.get("target", "")),
        "constraints": score_constraints(pred_task.get("constraints", []), gt_task.get("constraints", [])),
        "params": score_params(pred_task.get("params", {}), gt_task.get("params", {})),
    }

    scores["total"] = (
        0.35 * scores["action"] +
        0.30 * scores["target"] +
        0.15 * scores["constraints"] +
        0.20 * scores["params"]
    )
    return scores



/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ============================================================
# 18. main
# ============================================================

# --------------------------------------------------------
# GT読み込み
# --------------------------------------------------------
with open(GT_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# --------------------------------------------------------
# 候補辞書構築
# --------------------------------------------------------
global_dict = build_candidate_dictionary(dataset)

# --------------------------------------------------------
# モデルロード
# --------------------------------------------------------
tokenizer, model = load_model()

# --------------------------------------------------------
# 推論 + 評価
# --------------------------------------------------------
results = []
progress = tqdm(dataset, desc="infer+eval", total=len(dataset))

for idx, item in enumerate(progress, start=1):
    instruction = item["instruction"]

    pred = generate_task_json_ver08(
        instruction=instruction,
        dataset=dataset,
        global_dict=global_dict,
        tokenizer=tokenizer,
        model=model
    )

    score = score_prediction(pred, item)

    results.append({
        "video_path": item.get("video_path", ""),
        "instruction": instruction,
        "pred": pred,
        "score": score
    })

    progress.set_postfix({
        "idx": idx,
        "total": f"{score['total']:.3f}",
        "a": f"{score['action']:.3f}",
        "t": f"{score['target']:.3f}",
        "p": f"{score['params']:.3f}"
    })

# --------------------------------------------------------
# summary
# --------------------------------------------------------
keys = ["action", "target", "constraints", "params", "total"]
summary = {k: sum(r["score"][k] for r in results) / len(results) for k in keys}

print("\n===== SUMMARY =====")
print(json.dumps(summary, indent=2, ensure_ascii=False))

# --------------------------------------------------------
# 保存
# --------------------------------------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "summary": summary,
        "results": results
    }, f, ensure_ascii=False, indent=2)

print(f"\nSaved to: {OUTPUT_PATH}")

infer+eval: 100%|██████████| 100/100 [03:05<00:00,  1.85s/it, idx=100, total=0.432, a=0.423, t=0.171, p=0.414]


===== SUMMARY =====
{
  "action": 0.28518202837067036,
  "target": 0.16999517776580586,
  "constraints": 0.845,
  "params": 0.2907317064739831,
  "total": 0.33570860455427315
}

Saved to: /workspace/notebook/prediction_results_ver08.json
